In [3]:
%cd ..

c:\Users\Acer\mlops-kredit-sumatera


c:\Users\Acer\mlops-kredit-sumatera\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
from pathlib import Path

print(Path.cwd())

c:\Users\Acer\mlops-kredit-sumatera


In [5]:
import pandas as pd

df = pd.read_csv("data/raw/kredit_sumatera.csv")

print("Jumlah baris dan kolom:", df.shape)
display(df.head())


Jumlah baris dan kolom: (32400, 19)


,pengajuan_id,tanggal_pengajuan,periode,segmen,wilayah,kanal,usia,tanggungan,lama_usaha_tahun,penghasilan_bulanan,ragam_penghasilan,plafon_diajukan,tenor_bulan,bunga_tahunan,angsuran_bulanan,rata_saldo_90h,jumlah_tunggakan_12b,gagal_bayar,status_restrukturisasi
0,2022-01-00302,2022-01-01,2022-01,wiraswasta,Pesawaran,agen,28,3,6.2,6760000.0,0.282,14560000.0,12,0.0990,1333000.0,1231000.0,1,0,NaN
1,2022-01-00769,2022-01-01,2022-01,wiraswasta,Pringsewu,agen,48,1,2.4,3950000.0,0.226,11800000.0,18,0.1460,799000.0,755000.0,0,0,NaN
2,2022-01-00235,2022-01-01,2022-01,petani,Pringsewu,agen,44,4,2.2,4000000.0,0.428,4760000.0,36,0.1916,208000.0,10516000.0,0,0,NaN
3,2022-01-00661,2022-01-01,2022-01,petani,Pesawaran,cabang,45,1,2.9,7740000.0,0.444,32550000.0,36,0.1848,1405000.0,3295000.0,1,0,NaN
4,2022-01-00325,2022-01-01,2022-01,wiraswasta,Lampung Barat,cabang,46,2,28.2,11000000.0,0.363,26250000.0,18,0.1452,1776000.0,23195000.0,0,0,NaN


In [6]:

print("Informasi Dataset:")
df.info()

print("\nNama kolom:")
print(df.columns.tolist())

print("\nJumlah data duplikat:")
print(df.duplicated().sum())


Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32400 entries, 0 to 32399
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   pengajuan_id            32400 non-null  object 
 1   tanggal_pengajuan       32400 non-null  object 
 2   periode                 32400 non-null  object 
 3   segmen                  32400 non-null  object 
 4   wilayah                 32400 non-null  object 
 5   kanal                   32400 non-null  object 
 6   usia                    32400 non-null  int64  
 7   tanggungan              32400 non-null  int64  
 8   lama_usaha_tahun        31446 non-null  float64
 9   penghasilan_bulanan     32400 non-null  float64
 10  ragam_penghasilan       32400 non-null  float64
 11  plafon_diajukan         32400 non-null  float64
 12  tenor_bulan             32400 non-null  int64  
 13  bunga_tahunan           32400 non-null  float64
 14  angsuran_bulanan   

In [7]:

print("Tingkat gagal bayar:",
      f"{df.gagal_bayar.mean():.2%}")


Tingkat gagal bayar: 8.30%


In [8]:

print(
    pd.crosstab(
        df.periode.str[:4],
        df.segmen,
        normalize="index"
    ).round(3)
)


segmen   nelayan  pedagang_musiman  pegawai_tetap  petani  wiraswasta
periode                                                              
2022       0.067             0.139          0.421    0.13       0.242
2023       0.067             0.142          0.419    0.13       0.241
2024       0.099             0.171          0.339    0.16       0.231


In [9]:

hilang = df.isna().mean().sort_values(
    ascending=False
)

print(hilang[hilang > 0].round(4))

print(
    df.groupby("segmen")["rata_saldo_90h"]
      .apply(lambda s: s.isna().mean())
      .round(3)
)


status_restrukturisasi    0.9479
lama_usaha_tahun          0.0294
rata_saldo_90h            0.0231
dtype: float64
segmen
nelayan             0.108
pedagang_musiman    0.000
pegawai_tetap       0.000
petani              0.105
wiraswasta          0.000
Name: rata_saldo_90h, dtype: float64


In [10]:

print(
    df.groupby("gagal_bayar")["status_restrukturisasi"]
      .apply(lambda s: s.notna().mean())
)


gagal_bayar
0    0.000000
1    0.627509
Name: status_restrukturisasi, dtype: float64


In [11]:

df["rasio_angsuran"] = (
    df.angsuran_bulanan / df.penghasilan_bulanan
)

print(df[[
    "angsuran_bulanan",
    "penghasilan_bulanan",
    "rasio_angsuran"
]].head())


   angsuran_bulanan  penghasilan_bulanan  rasio_angsuran
0         1333000.0            6760000.0        0.197189
1          799000.0            3950000.0        0.202278
2          208000.0            4000000.0        0.052000
3         1405000.0            7740000.0        0.181525
4         1776000.0           11000000.0        0.161455


In [12]:

num = [
    "usia",
    "tanggungan",
    "lama_usaha_tahun",
    "penghasilan_bulanan",
    "ragam_penghasilan",
    "plafon_diajukan",
    "tenor_bulan",
    "bunga_tahunan",
    "rasio_angsuran",
    "rata_saldo_90h",
    "jumlah_tunggakan_12b"
]

X = df[num].join(
    pd.get_dummies(df[["segmen", "wilayah", "kanal"]])
)

y = df.gagal_bayar

print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)


Ukuran X: (32400, 24)
Ukuran y: (32400,)


In [13]:

latih = df.periode <= "2023-12"
uji = df.periode > "2024-06"

print("Jumlah data latih:", latih.sum())
print("Jumlah data uji:", uji.sum())


Jumlah data latih: 21600
Jumlah data uji: 5400


In [14]:

from sklearn.ensemble import HistGradientBoostingClassifier

m = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.06,
    max_iter=250,
    random_state=42
)

m.fit(X[latih], y[latih])

print("Model berhasil dilatih!")


c:\Users\Acer\mlops-kredit-sumatera\.venv\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Acer\mlops-kredit-sumatera\.venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\Acer\AppData\Local\Python\pythoncore-3.10-64\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\Acer\AppData\Local\Python\pythoncore-3.10-64\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Acer\AppData\Local\Python\pythoncore-3.10-64\lib\subprocess.py", line 1456, in _ex

Model berhasil dilatih!


In [15]:

from sklearn.metrics import roc_auc_score

skor = m.predict_proba(X[uji])[:, 1]

auc_oot = roc_auc_score(y[uji], skor)

print("AUC out-of-time:", round(auc_oot, 4))


AUC out-of-time: 0.7744


In [16]:

from sklearn.model_selection import train_test_split

Xa, Xb, ya, yb = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data latih acak:", len(Xa))
print("Jumlah data uji acak:", len(Xb))


Jumlah data latih acak: 25920
Jumlah data uji acak: 6480


In [17]:

m2 = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.06,
    max_iter=250,
    random_state=42
)

m2.fit(Xa, ya)

skor_acak = m2.predict_proba(Xb)[:, 1]

auc_acak = roc_auc_score(yb, skor_acak)

print("AUC pembagian acak:", round(auc_acak, 4))


AUC pembagian acak: 0.7631


In [18]:

print("=== PERBANDINGAN MODEL ===")
print("AUC Out-of-Time :", round(auc_oot, 4))
print("AUC Random Split:", round(auc_acak, 4))
print("Selisih AUC     :", round(auc_acak - auc_oot, 4))


=== PERBANDINGAN MODEL ===
AUC Out-of-Time : 0.7744
AUC Random Split: 0.7631
Selisih AUC     : -0.0112
